In [48]:
import pandas as pd
import json



In [61]:

path = "/content/users.csv"   # change path if needed

raw = pd.read_csv(path, header=None)

users = raw[0].str.split(",", expand=True)

users.columns = users.iloc[0]
users = users.drop(index=0).reset_index(drop=True)

users.head()

,user_id,age,gender,city,profession,personality_type,risk_appetite,relationship_status
0,1,27,M,Bangalore,Software Engineer,Analytical,High,Single
1,2,33,F,Mumbai,Marketing Manager,Expressive,Medium,Married
2,3,41,M,Delhi,Operations Lead,Practical,Low,Married
3,4,24,F,Pune,Student,Dreamer,Medium,Single
4,5,36,F,Hyderabad,Product Manager,Analytical,High,In a relationship


In [60]:
path = "/content/events.csv"

raw = pd.read_csv(path, header=None)

events = raw[0].str.split(",", expand=True)

events.columns = events.iloc[0]
events = events.drop(index=0).reset_index(drop=True)

events.head()


,event_id,user_id,event_type,event_intensity,event_recency_days,description
0,101,1,Work_Stress,4,3,"""Tight deadline and late nights for product la..."
1,102,1,Financial_Decision,3,15,"""Considering switching job for higher salary"""
2,103,2,Family_Conflict,2,7,"""Minor disagreement with spouse about work lif..."
3,104,2,Career_Opportunity,4,20,"""Offered a lateral move to a new brand team"""
4,105,3,Health_Concern,5,5,"""Doctor advised to reduce blood pressure and s..."


In [62]:
path = "/content/guidance_rules.csv"

raw = pd.read_csv(path, header=None)

rules = raw[0].str.split(",", expand=True)

rules.columns = rules.iloc[0]
rules = rules.drop(index=0).reset_index(drop=True)

rules.head()


,rule_id,archetype,condition_type,condition_value,recommended_category,priority,template_message,None,None
0,R1,Ambitious_Thinker,profession,Software Engineer,career,5,"""You are in a phase where focused effort on de...",None,None
1,R2,Ambitious_Thinker,risk_appetite,High,career,4,"""Your appetite for calculated risk is an asset...",None,None
2,R3,Balanced_Caretaker,personality_type,Nurturer,relationships,5,"""Your strength is providing emotional support....",None,None
3,R4,Health_Rebuilder,event_type,Health_Concern,health,5,"""Recent health signals are a nudge to reset yo...",movement,"or food."""
4,R5,Stressed_Professional,event_type,Work_Stress,health,4,"""High work pressure is not permanent but ignor...",None,None


In [63]:
print(users.columns)
print(events.columns)
print(rules.columns)


Index(['user_id', 'age', 'gender', 'city', 'profession', 'personality_type',
       'risk_appetite', 'relationship_status'],
      dtype='object', name=0)
Index(['event_id', 'user_id', 'event_type', 'event_intensity',
       'event_recency_days', 'description'],
      dtype='object', name=0)
Index([             'rule_id',            'archetype',       'condition_type',
            'condition_value', 'recommended_category',             'priority',
           'template_message',                   None,                   None],
      dtype='object', name=0)


In [64]:
def match_rules(user, user_events):
    matched = []

    for r in rules_records:

        # profile-based rules
        if r["condition_type"] in user:
            if str(user[r["condition_type"]]).strip() == str(r["condition_value"]).strip():
                matched.append(r)

        # event-based rules
        if r["condition_type"] == "event_type":
            if any(e["event_type"] == r["condition_value"] for e in user_events):
                matched.append(r)

    return matched


In [66]:
users_records = users.to_dict(orient="records")
events_records = events.to_dict(orient="records")
rules_records = rules.to_dict(orient="records")


/tmp/ipython-input-49556983.py:3: UserWarning: DataFrame columns are not unique, some columns will be omitted.
  rules_records = rules.to_dict(orient="records")


In [67]:
combined = []

for u in users_records:
    user_events = [e for e in events_records if e["user_id"] == u["user_id"]]
    matched = match_rules(u, user_events)

    combined.append({
        "user_id": u["user_id"],
        "profile": u,
        "events": user_events,
        "guidance": [
            {
                "rule_id": m["rule_id"],
                "archetype": m["archetype"],
                "category": m["recommended_category"],
                "priority": int(m["priority"]),
                "message": m["template_message"]
            }
            for m in matched
        ]
    })


In [68]:
import json

with open("/content/final_guidance.json", "w", encoding="utf-8") as f:
    json.dump(combined, f, indent=2)
